In [ ]:
import numpy as np

# 1. LOAD DATA (comma-separated)
# --------------------------------
# with open("tof.txt", "r") as f:
#     frames_flat = [list(map(float, line.strip().split(','))) for line in f.readlines()]
# frames_np = np.array(frames_flat)         # shape (n_frames, 64)
# frames_np = frames_np.reshape(-1, 8, 8)   # shape (n_frames, 8, 8)
# n_frames = frames_np.shape[0]
# frames_flat = []
# timestamps = []
with open("tof.txt", "r") as f:
    lines = [line.strip() for line in f]
    for i in range(0, len(lines)-1, 2):  # Step by 2
        ts = lines[i]
        data = lines[i+1]
        values = data.split(',')
        if len(values) == 64:
            frames_flat.append([float(v) for v in values])
            timestamps.append(ts)
        else:
            print(f"Bad data at line {i+2} with {len(values)} values.")

frames_np = np.array(frames_flat)
frames_np = frames_np.reshape(-1, 8, 8)
n_frames = frames_np.shape[0]
print(f"Good frames loaded: {n_frames}")
# 2. SIMULATE RANDOM DRINKING EVENTS & RANDOM VOLUMES
# ----------------------------------------------------
n_events = 10   # number of random "sip" events
clip_len = 16   # frames per clip
np.random.seed(0)
event_starts = np.sort(np.random.choice(np.arange(0, n_frames-clip_len, 30), n_events, replace=False))
event_ends = event_starts + np.random.randint(clip_len*2, clip_len*4, size=n_events)
event_ends = np.minimum(event_ends, n_frames-1)
event_volumes = np.round(np.random.uniform(20, 100, size=n_events), 1)   # mL

events = list(zip(event_starts, event_ends, event_volumes))

# 3. SLICE DATA INTO CLIPS
# ------------------------
stride = 4
clips = []
clip_start_idxs = []
for start in range(0, n_frames - clip_len + 1, stride):
    clips.append(frames_np[start:start+clip_len])
    clip_start_idxs.append(start)
clips = np.array(clips)     # (n_clips, clip_len, 8, 8)
clip_start_idxs = np.array(clip_start_idxs)
clip_end_idxs = clip_start_idxs + clip_len - 1
n_clips = clips.shape[0]

# 4. LABEL EACH CLIP WITH VOLUME (regression target)
# --------------------------------------------------
labels = np.zeros(n_clips, dtype=np.float32)   # default: 0 mL (no sip)
for i in range(n_clips):
    for ev_start, ev_end, ev_vol in events:
        overlap = (clip_end_idxs[i] >= ev_start) and (clip_start_idxs[i] <= ev_end)
        if overlap:
            labels[i] = ev_vol
            break

# 5. PREPARE DATA FOR ML MODEL
# ----------------------------
# Model expects (batch, time, height, width, channel)
clips = clips[..., None]  # add channel dim: shape (n_clips, 16, 8, 8, 1)

# Train/valid split (90/10)
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(clips, labels, test_size=0.1, random_state=42)

# 6. BUILD & TRAIN SIMPLE KERAS REGRESSION MODEL
# ----------------------------------------------
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Simple 3D CNN model for regression
model = keras.Sequential([
    layers.Input(shape=(clip_len, 8, 8, 1)),
    layers.Conv3D(16, (3, 3, 3), activation='relu', padding='same'),
    layers.MaxPooling3D((2, 2, 2)),
    layers.Conv3D(32, (3, 3, 3), activation='relu', padding='same'),
    layers.MaxPooling3D((2, 2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # regression: output=volume in mL
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# 7. TRAIN
# ---------
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=5, batch_size=16)

# 8. PRINT SAMPLE PREDICTIONS
# ---------------------------
y_pred = model.predict(X_val[:10]).flatten()
print("\nSample predictions (ml):")
for i in range(10):
    print(f" True: {y_val[i]:6.1f} | Predicted: {y_pred[i]:6.1f}")

# Done: This code shows the whole process
